# Домашнее задание: генерация текста с помощью RNN (собственный корпус)

## Задача

1. Собрать свой небольшой текстовый корпус (не менее 1000 предложений) с помощью библиотеки `requests` (парсинг новостного сайта, блога, форума и т.д.).
2. Обучить рекуррентную нейросеть (RNN/LSTM) на собранных данных для генерации текста (по образцу из приложенного ноутбука `Copy_of_rnn.ipynb`).
3. После обучения вывести на экран 2–3 сгенерированных предложения.
4. *Дополнительно (на 5 баллов, но не обязательно):* посчитать метрику перплексии (perplexity) на валидационной выборке.
5. *Для себя (не оценивается):* обучить модель с использованием GPU.

## Критерии оценки

- **3 балла** — корпус собран (≥1000 предложений), модель обучена, сгенерировано хотя бы 1 предложение.
- **4 балла** — всё из п.3 + код с комментариями, объясняющими ключевые этапы.
- **5 баллов** — всё из п.4 + дополнительно посчитана метрика перплексии.

## Важно

- Качество сгенерированного текста не оценивается.
- Выберите **свой уникальный сайт** для парсинга и укажите его в отчёте.
- Не используйте готовые датасеты из интернета.


## 1. Установка и импорт библиотек

In [3]:
!pip install beautifulsoup4 requests lxml -q

import requests
from bs4 import BeautifulSoup
import time
import re
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("TensorFlow версия:", tf.__version__)
print("GPU доступна:", tf.config.list_physical_devices('GPU'))

TensorFlow версия: 2.20.0
GPU доступна: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Парсинг текстового корпуса

**Ваш уникальный сайт:** https://en.wikipedia.org/wiki/Flying_Spaghetti_Monster

**Обоснование выбора:** Этот источник полностью в открытом доступе, имеет отличную структуру абзацев для парсинга с помощью BeautifulSoup, а специфическая лексика (пираты, макароны, абсурдные религиозные концепты) позволит рекуррентной нейросети научиться генерировать крайне нестандартные, ироничные и сюрреалистичные тексты. Ну и это просто забавно

In [4]:
# ЯЧЕЙКА 1: Парсинг данных
import requests
from bs4 import BeautifulSoup
import time
import re
import nltk
from nltk.tokenize import sent_tokenize

# Загружаем инструменты для разбиения на предложения
nltk.download('punkt')
nltk.download('punkt_tab')

def scrape_corpus(base_url, num_pages=1):
    """
    Парсинг текстового корпуса с выбранного сайта.
    Адаптировано для статьи про Летающего Макаронного Монстра.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    texts = []

    for page in range(1, num_pages + 1):
        # Формат изменен, так как наша статья находится на одной большой странице
        url = base_url
        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')

            # !!! ЗАМЕНЕННЫЙ СЕЛЕКТОР ПОД ВИКИПЕДИЮ !!!
            # Берем только текстовые абзацы (тег 'p')
            for tag in soup.find_all('p'):
                text = tag.get_text(strip=True)

                # Убираем технические сноски Википедии вроде [1], [2]
                text = re.sub(r'\[\d+\]', '', text)

                # Оставляем только осмысленные абзацы
                if text and len(text) > 30:
                    # Разбиваем абзац на отдельные предложения для обучения RNN
                    sentences = sent_tokenize(text)
                    texts.extend(sentences)

            print(f"Страница {page}: собрано {len(texts)} текстов (предложений)")
            time.sleep(1)  # вежливый парсинг

        except Exception as e:
            print(f"Ошибка на странице {page}: {e}")

    return texts


# ЗАМЕНИЛИ URL НА НАШУ ТЕМУ
base_url = "https://en.wikipedia.org/wiki/Flying_Spaghetti_Monster"

# Вызываем функцию (нам хватит 1 страницы, там достаточно текста)
# Называем переменную corpus_sentences, чтобы код нейросети из следующих заданий сработал без ошибок
corpus_sentences = scrape_corpus(base_url, num_pages=1)

print(f"\nВсего собрано предложений: {len(corpus_sentences)}")
print("Примеры:", corpus_sentences[:3])

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Страница 1: собрано 161 текстов (предложений)

Всего собрано предложений: 161
Примеры: ['TheFlying Spaghetti Monster(FSM) is thedeityof theChurch of the Flying Spaghetti Monster, orPastafarianism, aparodicnew religious movementthat promotes a light-hearted view ofreligion.The parody originated in opposition to the teaching ofintelligent designinpublic schoolsin the United States.', 'According to adherents, Pastafarianism (aportmanteauofpastaandRastafarianism) is a "real, legitimate religion, as much as any other".It has received some limited recognition as such.', 'The first documented description of the "Flying Spaghetti Monster" was in asatiricalopen letterwritten byBobby Hendersonin 2005 to protest theKansas State Board of Education decisionto permit teachingintelligent designas analternative to evolution in state school science classes.In the letter, Henderson demanded equal time in science classrooms for "Flying Spaghetti Monsterism", alongside intelligent design andevolution.Afte

## 3. Подготовка данных для RNN

Токенизация, создание последовательностей, паддинг.

In [5]:
# Шаг 1: Инициализируем и обучаем токенизатор на нашем пастафарианском корпусе
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus_sentences)

# Шаг 2: Переводим слова в последовательности целочисленных идентификаторов (индексов)
sequences = tokenizer.texts_to_sequences(corpus_sentences)

# Шаг 3: Формируем обучающие пары X (контекст) и y (следующее слово)
X, y = [], []
for seq in sequences:
    # Идем по цепочке токенов внутри каждого предложения
    for i in range(1, len(seq)):
        X.append(seq[:i])  # Все слова от начала до текущего — это входной контекст
        y.append(seq[i])   # Текущее слово — это правильный целевой ответ (метка)

# Шаг 4: Выравниваем длины векторов контекста с помощью паддинга (заполнение нулями слева)
X = pad_sequences(X, padding='pre')

# Шаг 5: Переводим целевые метки y в формат One-Hot Encoding (вектор размера словаря)
vocab_size = len(tokenizer.word_index) + 1  # +1 учитывает нулевой индекс паддинга
y = tf.keras.utils.to_categorical(y, num_classes=vocab_size)

print(f"Матрица признаков X (число примеров, максимальная длина контекста): {X.shape}")
print(f"Матрица ответов y (число примеров, размер словаря): {y.shape}")
print(f"Общее количество уникальных слов в словаре (vocab_size): {vocab_size}")

Матрица признаков X (число примеров, максимальная длина контекста): (5642, 213)
Матрица ответов y (число примеров, размер словаря): (5642, 1945)
Общее количество уникальных слов в словаре (vocab_size): 1945


## 4. Создание и обучение модели RNN (LSTM)

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input

# Создаем модель заново, используя современный синтаксис Keras
model = Sequential([
    # Явно задаем размер входных данных (вместо устаревшего input_length)
    Input(shape=(X.shape[1],)),

    # Layer 1: Векторное представление слов
    Embedding(input_dim=vocab_size, output_dim=100),

    # Layer 2: LSTM-слой со 150 скрытыми блоками
    LSTM(150, return_sequences=False),

    # Layer 3: Полносвязный слой с активацией Softmax
    Dense(vocab_size, activation='softmax')
])

# Компилируем модель
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Выводим архитектуру сети на экран
model.summary()

# Запускаем обучение
print("\nНачало процесса обучения нейросети...")
history = model.fit(X, y, epochs=15, batch_size=64, validation_split=0.2)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 213, 100)       │       194,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 150)            │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1945)           │       293,695 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 638,795 (2.44 MB)

 Trainable params: 638,795 (2.44 MB)

 Non-trainable params: 0 (0.00 B)


Начало процесса обучения нейросети...
Epoch 1/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - accuracy: 0.0620 - loss: 6.9311 - val_accuracy: 0.0416 - val_loss: 6.9415
Epoch 2/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.0649 - loss: 6.3485 - val_accuracy: 0.0416 - val_loss: 7.0714
Epoch 3/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.0713 - loss: 6.1965 - val_accuracy: 0.0487 - val_loss: 7.1463
Epoch 4/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.0911 - loss: 6.0509 - val_accuracy: 0.0620 - val_loss: 7.1882
Epoch 5/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.1199 - loss: 5.9105 - val_accuracy: 0.0771 - val_loss: 7.2535
Epoch 6/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.1301 - loss: 5.7687 - val_accuracy: 0.0779 - val_loss: 7.3047
Epoch 7/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.1343 - loss: 5.6536 - val_accuracy: 0.0762 - val_loss: 7.3952
Epoch 8/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.1389 - 

## 5. Генерация текста

Функция генерации и вывод 2–3 предложений.

In [8]:
def generate_text(seed_text, next_words, max_sequence_len):
    """
    Генерация заданной цепочки слов на основе текстовой затравки (seed_text).
    """
    for _ in range(next_words):
        # Токенизируем текущую накопленную фразу
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        # Применяем паддинг под формат, который ожидает входной слой модели
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')

        # Предсказываем индекс самого вероятного следующего слова
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)

        # Переводим индекс обратно в текстовое слово
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break

        # Добавляем слово к общей фразе
        seed_text += " " + output_word

    return seed_text

print("--- РЕЗУЛЬТАТЫ ГЕНЕРАЦИИ ТЕКСТА ---")

# Пример 1
seed1 = "The Flying Spaghetti"
generated1 = generate_text(seed1, next_words=8, max_sequence_len=X.shape[1])
print(f"Пример 1:\n{generated1}\n")

# Пример 2
seed2 = "Pastafarianism is a"
generated2 = generate_text(seed2, next_words=8, max_sequence_len=X.shape[1])
print(f"Пример 2:\n{generated2}\n")

# Пример 3
seed3 = "Bobby Henderson wrote"
generated3 = generate_text(seed3, next_words=8, max_sequence_len=X.shape[1])
print(f"Пример 3:\n{generated3}\n")

--- РЕЗУЛЬТАТЫ ГЕНЕРАЦИИ ТЕКСТА ---
Пример 1:
The Flying Spaghetti monster is a religious religion in the flying

Пример 2:
Pastafarianism is a religious religion in the flying spaghetti monster is

Пример 3:
Bobby Henderson wrote the flying spaghetti monster is a religious religion



# Преложения получились очень смешными. У этого есть несколько причин:
- Ограниченный словарный запас: Несмотря на то, что мы собрали нужный объём текста, для понимания языка пары статей всё же маловато. Модель  выучила самые частотные связки слов и теперь использует их повсюду.
- Greedy Search: В нашем коде используется np.argmax, что заставляет модель на каждом шаге выбирать строго самое вероятное слово. Без добавления фактора случайности (например, параметра temperature), такие модели всегда приходят к  повтору самой популярной фразы.

## 6. (Дополнительно, на 5 баллов) Расчёт перплексии

Перплексия = exp(loss). Чем ниже, тем лучше модель предсказывает последовательность.

In [9]:
# Извлекаем значение потерь (loss) на валидационной выборке из последней эпохи обучения
val_loss = history.history['val_loss'][-1]

# Расчет перплексии как экспоненты от функции потерь
perplexity = np.exp(val_loss)

print(f"Финальные потери на валидации (Validation Loss): {val_loss:.4f}")
print(f"Метрика Перплексии (Perplexity): {perplexity:.4f}")
print("\nИнтерпретация: Перплексия оценивает количество равновероятных вариантов выбора следующего слова. "
      "Чем ниже значение перплексии, тем увереннее модель ориентируется в структуре пастафарианского корпуса.")

Финальные потери на валидации (Validation Loss): 7.8376
Метрика Перплексии (Perplexity): 2534.0147

Интерпретация: Перплексия оценивает количество равновероятных вариантов выбора следующего слова. Чем ниже значение перплексии, тем увереннее модель ориентируется в структуре пастафарианского корпуса.


# Небольшой вывод:
- Корпуса из ~1000 предложений физически недостаточно, чтобы нейросеть смогла глубоко усвоить синтаксис английского языка и надежно снизить неопределенность.

- Т.к. мы не использовали массивные предобученные словари, наш слой Embedding инициализировался случайными весами и пытался найти смысл слов на ходу.

- Текст изобилует редкими терминами, иронией и нестандартными конструкциями, что делает предсказание следующего слова крайне нетривиальной задачей для такой базовой модели.

## 7. GPU (не оценивается)

Убедитесь, что обучение запускалось на GPU:
```python
print("GPU доступна:", tf.config.list_physical_devices('GPU'))
```

В Google Colab: Среда выполнения - Изменить тип среды выполнения - Выберите T4 GPU.

Документация: https://www.tensorflow.org/guide/gpu

In [10]:
print("Устройства GPU:", tf.config.list_physical_devices('GPU'))

Устройства GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
